# Fine-tuning transformers for Finnish

Let's fine-tune four transformer models for Finnish and compare their performance! The goal of this notebook is to fine-tune pretrained models from Hugging Face and evaluate how well they perform on an intent recognition task.

In Notebook 1, we explored and compared tokenization methods for Finnish. Building on that work, we will now move on to fine-tuning transformer models and evaluating their results.

## Install dependencies

In [ ]:
pip install transformers datasets evaluate pandas

## Load data

We will load the dataset from Hugging Face: [mteb/amazon_massive_intent](https://huggingface.co/datasets/mteb/amazon_massive_intent). This multilingual dataset contains user utterances annotated with intent labels, making it suitable for intent classification. It is already split into training, validation, and test sets, which we will use to fine-tune our models, monitor their performance, and evaluate their results.

In [ ]:
from datasets import load_dataset

# load multilingual dataset
dataset = load_dataset("mteb/amazon_massive_intent")

# keep only Finnish examples
hf_train_dataset = dataset["train"].filter(
    lambda example: example["lang"] == "fi"
)

hf_validation_dataset = dataset["validation"].filter(
    lambda example: example["lang"] == "fi"
)

hf_test_dataset = dataset["test"].filter(
    lambda example: example["lang"] == "fi"
)

In [3]:
print("Total number of training samples:", len(hf_train_dataset), "and number of classes:", len(set(hf_train_dataset["label"])))
print("Total number of validation samples", len(hf_validation_dataset), "and number of classes:", len(set(hf_validation_dataset["label"])))
print("Total number of test samples", len(hf_test_dataset), "and number of classes:", len(set(hf_test_dataset["label"])))

Total number of training samples: 11514 and number of classes: 60
Total number of validation samples 2033 and number of classes: 59
Total number of test samples 2974 and number of classes: 59


The dataset contains 60 intent classes, which makes this a fairly complex classification task. To keep things manageable in this first iteration, we will simplify the task by selecting 10 classes and training our models to distinguish between them.

In [4]:
from collections import Counter

# Find the 10 most frequent classes in the Finnish TRAINING data
class_counts = Counter(hf_train_dataset["label"])

top_10_classes = [
    label for label, count in class_counts.most_common(10)
]

print("Top 10 Finnish classes:", top_10_classes)
print("Number of Finnish training examples:", len(hf_train_dataset))

Top 10 Finnish classes: ['calendar_set', 'play_music', 'weather_query', 'calendar_query', 'general_quirky', 'qa_factoid', 'news_query', 'email_query', 'email_sendemail', 'datetime_query']
Number of Finnish training examples: 11514


Let's filter the training, validation, and test sets to keep only samples belonging to our selected 10 intent classes. This ensures that all three splits use the same set of labels.

In [ ]:
# Keep ONLY those 10 classes in all three splits
hf_train_dataset_final = hf_train_dataset.filter(
    lambda example: example["label"] in top_10_classes
)

hf_val_dataset_final = hf_validation_dataset.filter(
    lambda example: example["label"] in top_10_classes
)

hf_test_dataset_final = hf_test_dataset.filter(
    lambda example: example["label"] in top_10_classes
)

In [6]:
# verify that everything is Finnish and only contains the 10 classes
print("Train languages:", set(hf_train_dataset_final["lang"]))
print("Validation languages:", set(hf_val_dataset_final["lang"]))
print("Test languages:", set(hf_test_dataset_final["lang"]))
print("----------------------")
print("Train classes:", set(hf_train_dataset_final["label"]))
print("Validation classes:", set(hf_val_dataset_final["label"]))
print("Test classes:", set(hf_test_dataset_final["label"]))
print("----------------------")
print("Train samples:", len(hf_train_dataset_final))
print("Validation samples:", len(hf_val_dataset_final))
print("Test samples:", len(hf_test_dataset_final))

Train languages: {'fi'}
Validation languages: {'fi'}
Test languages: {'fi'}
----------------------
Train classes: {'email_sendemail', 'calendar_query', 'calendar_set', 'email_query', 'general_quirky', 'datetime_query', 'weather_query', 'play_music', 'news_query', 'qa_factoid'}
Validation classes: {'email_sendemail', 'calendar_query', 'calendar_set', 'email_query', 'general_quirky', 'datetime_query', 'weather_query', 'play_music', 'news_query', 'qa_factoid'}
Test classes: {'email_sendemail', 'calendar_query', 'calendar_set', 'email_query', 'general_quirky', 'datetime_query', 'weather_query', 'play_music', 'news_query', 'qa_factoid'}
----------------------
Train samples: 5312
Validation samples: 959
Test samples: 1422


We need to convert the intent labels into integers from 0 to 9 so that our models can use them for classification. We'll also create an `id2label` mapping, which lets us translate the model's numerical predictions back into the original intent names. This will make the results easier to interpret when evaluating our fine-tuned models.


In [7]:
label2id = {
    "calendar_query": 0,
    "calendar_set": 1,
    "datetime_query": 2,
    "email_query": 3,
    "email_sendemail": 4,
    "general_quirky": 5,
    "news_query": 6,
    "play_music": 7,
    "qa_factoid": 8,
    "weather_query": 9,
}

id2label = {v: k for k, v in label2id.items()}

def map_labels(example):
    example["labels"] = label2id[example["label"]]
    return example

In [ ]:
hf_train_dataset_final = hf_train_dataset_final.map(map_labels)
hf_val_dataset_final = hf_val_dataset_final.map(map_labels)
hf_test_dataset_final = hf_test_dataset_final.map(map_labels)

Let's verify that all labels have been mapped correctly and that the dataset contains only integer labels from 0 to 9.

In [9]:
print(sorted(set(hf_train_dataset_final["labels"])))
print(sorted(set(hf_val_dataset_final["labels"])))
print(sorted(set(hf_test_dataset_final["labels"])))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## Tokenize dataset

Before fine-tuning our models, we need to tokenize the text. We will apply **truncation** to limit the length of each input sequence to a specified maximum. A common choice is 512 tokens, although models such as ModernBERT support sequences of up to 8,192 tokens.

For now, we will leave **padding** to a later stage, where we can handle it during batching.

After tokenization, we also need to add the labels to the data. Each example should therefore contain the following elements:

```
{
    "input_ids": [101, 2054, 2003, ...],
    "attention_mask": [1, 1, 1, ...],
    "token_type_ids": [0, 0, 0, ...],
    "labels": 9
}
```

The exact token IDs will depend on the tokenizer used by each model.

In [10]:
MAX_LENGTH = 512

In [11]:
from datasets import Dataset

def _tokenize_function_helper(examples, tokenizer, truncation, max_length):
    kwargs = {
        "truncation": truncation,
        "padding": False
    }
    if max_length is not None:
        kwargs["max_length"] = max_length

    tokenized_inputs = tokenizer(
        examples["text"],
        **kwargs
    )
    # Explicitly add the 'labels' column back to the tokenized output
    tokenized_inputs["labels"] = examples["labels"]
    return tokenized_inputs

def tokenize_dataset(dataset, tokenizer, max_length=None):
    """
    Tokenizes the dataset.

    Sequences longer than max_length are truncated.
    No padding is performed here; padding should be handled dynamically
    when batches are created.
    """

    # Use remove_columns to ensure only tokenized inputs and labels are kept
    return dataset.map(
        lambda examples: _tokenize_function_helper(examples, tokenizer, True, max_length),
        batched=True,
        remove_columns=['text', 'label', 'lang'] # Remove original text, label, and lang columns
    )

In [ ]:
from transformers import AutoTokenizer

tokenizer_finbert = AutoTokenizer.from_pretrained("TurkuNLP/bert-base-finnish-cased-v1")
tokenizer_finnish_roberta = AutoTokenizer.from_pretrained("Finnish-NLP/roberta-large-finnish-v2")
tokenizer_finnish_modernbert = AutoTokenizer.from_pretrained("TurkuNLP/finnish-modernbert-base")
tokenizer_xlm_r = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")

In [ ]:
tokenizers_and_names = [
    (tokenizer_finbert, "finbert"),
    (tokenizer_finnish_roberta, "finnish_roberta"),
    (tokenizer_finnish_modernbert, "finnish_modernbert"),
    (tokenizer_xlm_r, "xlm_r")
]

for tokenizer_obj, name in tokenizers_and_names:
    globals()[f"tokenized_train_{name}"] = tokenize_dataset(hf_train_dataset_final, tokenizer_obj)
    globals()[f"tokenized_validation_{name}"] = tokenize_dataset(hf_val_dataset_final, tokenizer_obj)
    globals()[f"tokenized_test_{name}"] = tokenize_dataset(hf_test_dataset_final, tokenizer_obj)

## Fine-tune the transformer
Let's fine-tune four transformer models and compare their performance on our Finnish intent classification task.

Before training, there are two important things to consider when preparing the input data:

- **Model-specific tokenization**: We need to tokenize the data using the tokenizer associated with each pretrained model. The tokenizer converts text into token IDs that correspond to the model's vocabulary and learned embeddings. Using an incompatible tokenizer can result in incorrect input representations, so each model must use its own tokenizer.
- **Dynamic padding**: In addition to tokenizing and truncating the text, we need to ensure that sequences in the same batch have the same length. We will use dynamic padding, which pads sequences to the length of the longest sequence in each batch. This avoids unnecessary padding and helps reduce computational overhead.

A few practical notes before we begin:

> **GPU recommended:** If you're using Google Colab, switch to a GPU runtime to speed up training.

> **Storage considerations:** Each model checkpoint may occupy approximately 1.3 GB with the parameters used in this notebook. Make sure you have enough storage available, especially if you save checkpoints for all four models.

###Configuration

In [14]:
# config
NUM_LABELS = 10
MODEL_NAME = "TurkuNLP/finnish-modernbert-base"

TRAIN_DATASET = tokenized_train_finnish_modernbert
VAL_DATASET = tokenized_validation_finnish_modernbert
TEST_DATASET = tokenized_test_finnish_modernbert

PER_DEVICE_TRAIN_BATCH_SIZE = 16
NUM_TRAIN_EPOCHS = 3
WEIGHT_DECAY = 0.01
LEARNING_RATE = 2e-5
EVALUATION_STRATEGY = "epoch"

# Automatically use the model name in the output directory
MODEL_KEY = MODEL_NAME.split("/")[-1]
OUTPUT_DIR = f"./results/{MODEL_KEY}"

### Define evaluation metrics

We will define custom evaluation metrics for training, since the Hugging Face `Trainer` reports the loss by default. In addition to loss, we want to track **accuracy, precision, recall, and macro F1** to get a more complete picture of each model's performance.

Macro F1 is particularly useful because the 10 intent classes may have different numbers of samples. It calculates the F1 score separately for each class and then takes the unweighted average, giving every class equal importance regardless of how many samples it contains.

These metrics will help us compare the four models and assess how well they perform across different intent classes.


In [15]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "macro_f1": f1
    }

### Perform fine-tuning


In [16]:
from transformers import (
   AutoModelForSequenceClassification,
   Trainer,
   TrainingArguments,
   DataCollatorWithPadding
  )

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    label2id=label2id,
    id2label=id2label
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy=EVALUATION_STRATEGY ,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    weight_decay=WEIGHT_DECAY,
)

# add dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_finnish_modernbert)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=TRAIN_DATASET,
    eval_dataset=VAL_DATASET,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  574MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: TurkuNLP/finnish-modernbert-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Macro F1
1,No log,0.401979,0.870699,0.881730,0.866431,0.871078
2,0.523355,0.367739,0.887383,0.893462,0.883541,0.887861
3,0.523355,0.394937,0.891554,0.889438,0.889493,0.889112


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=996, training_loss=0.3173422717665094, metrics={'train_runtime': 138.7056, 'train_samples_per_second': 114.891, 'train_steps_per_second': 7.181, 'total_flos': 193273660758720.0, 'train_loss': 0.3173422717665094, 'epoch': 3.0})

## Test the model

Now let's evaluate our fine-tuned model on unseen data using the test set we prepared earlier. This will give us an indication of how well the model generalizes to new examples.

We'll also save the model's predictions, the true labels, and any other relevant information so that we can perform a more detailed error analysis in a separate notebook.


In [18]:
import os
import numpy as np
import pandas as pd

SAVE_DIR = f"./evaluation_results/{MODEL_NAME}"
os.makedirs(SAVE_DIR, exist_ok=True)

# Get predictions on the test set
test_results = trainer.predict(tokenized_test_finnish_modernbert)

# Predictions and true labels
y_pred = np.argmax(test_results.predictions, axis=1)
y_true = test_results.label_ids

# Actual texts
texts = hf_test_dataset_final["text"]

# Save everything needed for later analysis
results_df = pd.DataFrame({
    "text": texts,
    "true_label": y_true,
    "predicted_label": y_pred,
    "correct": y_true == y_pred
})

# Save readable predictions
results_df.to_csv(
    f"{SAVE_DIR}/predictions.csv",
    index=False
)

# Save raw logits too, in case you want confidence analysis later
np.save(
    f"{SAVE_DIR}/logits.npy",
    test_results.predictions
)

# Save evaluation metrics
metrics = test_results.metrics
pd.DataFrame([metrics]).to_csv(
    f"{SAVE_DIR}/metrics.csv",
    index=False
)

# Save label mapping
pd.DataFrame(
    list(id2label.items()),
    columns=["label_id", "label_name"]
).to_csv(
    f"{SAVE_DIR}/label_mapping.csv",
    index=False
)

print(f"Saved evaluation results to: {SAVE_DIR}")

Saved evaluation results to: ./evaluation_results/TurkuNLP/finnish-modernbert-base


##Conclusion

In this notebook, we fine-tuned four pretrained transformer models for Finnish intent classification using a subset of 10 classes from the massive dataset. We prepared the data, tokenized the inputs using each model's corresponding tokenizer, and evaluated the fine-tuned models on a held-out test set.

We also saved the evaluation results and model predictions for further analysis.

In the next notebook, we will compare the four models' performance using the evaluation metrics calculated here and visualize the results. We will also investigate their predictions to better understand their strengths, weaknesses, and common sources of error.